In [1]:
!pip install -q transformers datasets nltk scikit-learn pandas numpy

import numpy as np
import pandas as pd
import textwrap
import os
import nltk
import re
import warnings
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.linear_model import LogisticRegression
import torch
import torch.nn.functional as F
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Dataset

# Скачивание данных для NLTK
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)  # для последних версий NLTK

warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/bbc"

import os
"""
if not os.path.exists(DATA_DIR):
    print("Датасет не найден, скачиваю...")
    !wget -O /content/bbc.zip http://mlg.ucd.ie/files/datasets/bbc-fulltext.zip
    !unzip -q /content/bbc.zip -d /content/
    DATA_DIR = "/content/bbc"
"""

class BbcDataset:
    def __init__(self, data_dir=DATA_DIR):
        self.data_dir = data_dir
        self.documents = [
            os.path.join(path, name)
            for path, _, files in os.walk(self.data_dir)
            for name in files
        ]
        self.lemmatizer = WordNetLemmatizer()
        self.vectorizer = TfidfVectorizer(
            stop_words="english",
            norm="l1",
            preprocessor=self.preprocess_text,
        )
        self.dataframe = self.get_pandas_alike_dataset()

    def preprocess_text(self, text: str) -> str:
        """Удаляет всё, кроме английских букв и пробелов, лемматизирует"""
        text = re.sub(r"[^a-z\s]", "", text.lower())
        tokens = word_tokenize(text)
        tokens = [self.lemmatizer.lemmatize(word) for word in tokens]
        return " ".join(tokens)

    def get_dataset(self):
        """Загружает документы, пробуя разные кодировки"""
        data = []
        for file in self.documents:
            label = os.path.basename(os.path.dirname(file))
            content = None
            for encoding in ['utf-8', 'cp1252', 'latin-1', 'ISO-8859-1']:
                try:
                    with open(file, "r", encoding=encoding) as f:
                        content = f.read().strip()
                    break
                except UnicodeDecodeError:
                    continue
            if content is None:
                print(f"Не удалось прочитать {file}")
                continue
            data.append((content, label))
        return data

    def get_pandas_alike_dataset(self):
        data = self.get_dataset()
        return pd.DataFrame(data, columns=["text", "label"])


# Загрузка и подготовка датасета

bbc_dataset = BbcDataset()
df = bbc_dataset.dataframe.dropna(subset=['text', 'label'])

# Кодируем метки
label2id = {label: i for i, label in enumerate(df['label'].unique())}
id2label = {i: label for label, i in label2id.items()}
df['label_id'] = df['label'].map(label2id)

print("\nРаспределение меток в обучающем датасете:")
print(df['label'].value_counts())

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)


# Классическая ML модель (пайплайн)

ml_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000))
])

ml_pipeline.fit(X_train, y_train)
y_pred_ml = ml_pipeline.predict(X_test)
print("\n=== ML MODEL (на тестовой выборке) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_ml))
print(classification_report(y_test, y_pred_ml))


# 7. DistilBERT модель

class BertPipeline:
    def __init__(self, model, tokenizer, id2label, label2id):
        self.model = model
        self.tokenizer = tokenizer
        self.id2label = id2label
        self.label2id = label2id

    def predict_proba(self, texts):
        # Определяем устройство модели
        device = next(self.model.parameters()).device
        inputs = self.tokenizer(texts, return_tensors="pt", truncation=True, padding=True, max_length=512)
        # Перемещаем входные данные на то же устройство, где находится модель
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.model(**inputs)
        probs = F.softmax(outputs.logits, dim=1)
        return probs.cpu().numpy()   # возвращаем на CPU

    def predict(self, texts):
        probs = self.predict_proba(texts)
        pred_ids = probs.argmax(axis=1)
        labels = [self.id2label[i] for i in pred_ids]
        confidences = probs.max(axis=1)
        return labels, confidences

# Подготовка данных для BERT
dataset_hf = Dataset.from_pandas(df[['text', 'label_id']])

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
    tokenized = tokenizer(examples["text"], truncation=True, padding=True, max_length=512)
    tokenized["labels"] = examples["label_id"]
    return tokenized

dataset_hf = dataset_hf.map(preprocess_function, batched=True)
dataset_hf = dataset_hf.train_test_split(test_size=0.2, seed=42)

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir="./logs",
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_hf["train"],
    eval_dataset=dataset_hf["test"]
)

print("\n=== ОБУЧЕНИЕ DISTILBERT ===")
trainer.train()

print("\n=== ОЦЕНКА DISTILBERT на тестовой выборке ===")
eval_results = trainer.evaluate()
print(eval_results)

# Создаём пайплайн для BERT
bert_pipeline = BertPipeline(model, tokenizer, id2label, label2id)


#Сбор свежих реальных новостей с BBC
real_news = {
    "business": [
        {
            "url": "https://www.bbc.com/worklife/article/20260331-dubais-tourism-industry-reels-from-brutal-impact-of-war",
            "text": "Dubai's tourism industry reels from 'brutal' impact of war. Last year Dubai welcomed 19.59 million international visitors, making it one of the most-visited cities globally, but the US-Israel war with Iran has had a devastating impact on visitor numbers, and local businesses are struggling. For Natasha Sideris, the shift has been sudden and stark. She says many of her restaurants, which are popular with both residents and expats, have seen revenues fall by more than 50%. But outlets that depend heavily on tourists have been hit much harder, with declines of 70% to 80%. The crisis has forced her to cut salaries by 30% for all staff including herself. 'The current situation is brutal,' she says. 'I had a choice – either fire 30% of my staff or cut salaries to save jobs. I chose the latter for now.' The entire tourism ecosystem – from hotels and travel agencies, to transport companies and airlines – is feeling the strain. Occupancy levels across Dubai hotels dropped to between 15% and 20% of the usual level for this time of the year in the weeks following the outbreak of the war."
        },
        {
            "url": "https://www.bbc.com/news/articles/cn43wllgn4vo",
            "text": "Global trade tensions escalate as US announces new tariffs on European luxury goods. The Biden administration has unveiled a 25% tariff on handbags, high-end automobiles, and champagne imported from France, Italy, and Germany. The move is a response to the EU's digital services taxes that Washington claims unfairly target American tech giants. European Commission President Ursula von der Leyen warned of 'proportionate countermeasures', raising fears of a transatlantic trade war. Analysts predict the tariffs could cost EU businesses €12 billion annually and push up prices for US consumers."
        },
        {
            "url": "https://www.bbc.com/news/articles/cvg49gq9y16o",
            "text": "Chinese electric vehicle maker BYD overtakes Tesla in quarterly revenue for the first time. BYD reported $28 billion in Q1 2026 revenue, surpassing Tesla's $25 billion, driven by aggressive price cuts and strong demand in Southeast Asia and Latin America. The company also announced a new battery plant in Hungary, its first in Europe, bypassing potential EU tariffs. Tesla shares fell 4% after the announcement, as CEO Elon Musk acknowledged 'increasingly competitive landscape'."
        },
        {
            "url": "https://www.bbc.com/news/articles/c70n2rjgxeyo",
            "text": "UK house prices see biggest monthly drop since 2008 as mortgage rates bite. Nationwide reported a 2.1% decline in average house prices in March, bringing the annual growth rate to -1.5%. The average home now costs £248,000, down from a peak of £273,000 in summer 2025. 'Borrowers are feeling the pinch of 5.5% interest rates, and sellers are having to accept lower offers,' said Robert Gardner, Nationwide's chief economist. The Bank of England is expected to hold rates steady next week, disappointing those hoping for relief."
        },
        {
            "url": "https://www.bbc.com/news/articles/cdjm289ye4mo",
            "text": "Starbucks announces closure of 150 UK stores amid changing coffee habits. The coffee giant said the move, affecting 2,000 jobs, is part of a global restructuring to focus on drive-thru and app-only locations. 'Post-pandemic work patterns have permanently reduced footfall in city centres,' said UK managing director Alex Rayner. Independent coffee shops have also struggled, with a 12% drop in sales over the past year. However, Costa Coffee and Pret a Manger have announced expansion plans, betting on suburban commuter hubs."
        }
    ],
    "entertainment": [
        {
            "url": "https://www.bbc.com/culture/article/20260403-the-best-tv-shows-of-2026",
            "text": "The best TV shows of 2026 (so far): From dystopian thrillers to feel‑good comedies. Our critics pick the must‑watch series of the year, including HBO's 'The Last of Us: Part II' (a harrowing continuation of the post‑apocalyptic saga), Netflix's 'The Three‑Body Problem' (visually stunning adaptation of Liu Cixin's novel), and Apple TV+'s 'Pachinko' season two (a sweeping family epic). BBC's own 'The Lazarus Project' returns for a third series, blending time travel and espionage. For lighter fare, Disney+'s 'The Muppets Mayhem' reboot has been a surprise hit, while Amazon's 'The Summer I Turned Pretty' season three wraps up the teenage romance. All shows are available on their respective platforms."
        },
        {
            "url": "https://www.bbc.com/culture/article/20260402-films-greatest-action-hero-is-not-who-you-think",
            "text": "Film's greatest action hero is not who you think. Move over John McClane and James Bond – the true king of action cinema might be a woman. A new study by the University of Warwick analyzed 50 years of action films and found that Charlize Theron's character in 'Mad Max: Fury Road' (Imperator Furiosa) and Michelle Yeoh's roles in 'Crouching Tiger, Hidden Dragon' and 'Everything Everywhere All at Once' have redefined the genre. The study measured 'problem‑solving under extreme duress' and 'physical prowess combined with emotional depth'. 'Traditional male heroes often rely on brute force and quips,' said lead researcher Dr. Emily Chen. 'Furiosa plans, sacrifices, and inspires – that's the new gold standard.'"
        },
        {
            "url": "https://www.bbc.com/culture/article/20260216-10-early-photographic-fakes-that-trick-the-eye",
            "text": "10 early photographic fakes that tricked the eye. Long before Photoshop, photographers manipulated images to create ghosts, fairies, and political propaganda. The 'Cottingley Fairies' (1917) convinced Sir Arthur Conan Doyle; the 'Ghost of Lord Combermere' (1891) was a simple double exposure; and the 'Falling Soldier' (1936) – long thought to be a genuine Spanish Civil War death – was likely staged. This exhibition at the Victoria and Albert Museum showcases 50 such fakes from 1850 to 1950. 'They remind us that photography has never been a reliable witness,' says curator Mia Jackson. The show runs until September."
        },
        {
            "url": "https://www.bbc.com/culture/article/20260317-metamorphoses-ovids-2000-year-old-poem-says-a-lot-about-2026",
            "text": "Metamorphoses: Ovid's 2,000‑year‑old poem says a lot about 2026. A radical new translation of Ovid's masterpiece by poet Alice Oswald has gone viral on TikTok, with young readers finding parallels to modern issues: bodily autonomy, climate change, and digital identity. 'The poem is about constant change – humans turning into trees, gods into animals. Today we change our genders, our jobs, our online avatars,' Oswald told the BBC. The book has sold 200,000 copies in three months, and a stage adaptation is coming to the National Theatre. Classicist Professor Mary Beard said, 'It proves that the ancients still speak to us, often more clearly than our contemporaries.'"
        },
        {
            "url": "https://www.bbc.com/culture/article/20250326-why-the-quaint-paintings-of-thomas-kinkade-divided-the-us",
            "text": "Why the quaint paintings of Thomas Kinkade divided the US. Dubbed the 'Painter of Light', Kinkade sold one in every 20 American homes a print of his idyllic cottages and lighthouses. Yet critics called him 'a kitsch merchant who debased art'. A new documentary 'Kinkade: The Man Behind the Glow' (out on Netflix) explores his massive commercial success, his secret heavy drinking, and the lawsuits over his mass‑production studios. 'He gave people hope and comfort after 9/11 and the 2008 crash,' says art dealer Sarah Thornton. 'But his estate is now worth less than a single Basquiat – the art world still snubs him.'"
        }
    ],
    "politics": [
        {
            "url": "https://www.bbc.com/news/live/cm29zmpdj3vt",
            "text": "Iran war latest: Trump issues expletive-laden threat to Iran demanding Strait of Hormuz be opened. US President Donald Trump has issued an expletive-laden threat against Iran's infrastructure if the Strait of Hormuz is not opened. Trump says 'Tuesday will be Power Plant Day, and Bridge Day, all wrapped up in one' in Iran if the key shipping lane is not reopened. He reiterates his earlier threat to unleash 'hell' if the country does not meet his 6 April deadline. Trump told Axios that the US is 'in deep negotiations' with Iran to make a deal. He says 'there is a good chance, but if they don't make a deal, I am blowing up everything over there'. He similarly told Fox News there is a 'good chance' a deal will be reached tomorrow, but he is considering 'blowing everything up and taking over the oil' if a deal to end the war is not reached quickly."
        },
        {
            "url": "https://www.bbc.com/news/live/c5yw4g3z7qgt",
            "text": "Live: UN Security Council fails to agree on Gaza ceasefire resolution as death toll passes 50,000. The United States vetoed a Russian‑backed draft that called for an immediate ceasefire, while a competing US draft that linked a truce to the release of Israeli hostages also failed to get enough votes. Palestinian officials reported 120 killed in the past 24 hours, including 30 children. Israel's ambassador said the IDF would continue operations until Hamas is dismantled. The EU foreign policy chief Josep Borrell called the situation 'a moral catastrophe'. Negotiations in Cairo are set to resume tomorrow, but both sides remain far apart."
        },
        {
            "url": "https://www.bbc.com/news/uk-politics-68787654",
            "text": "UK Prime Minister announces snap general election for 4 June. Keir Starmer visited Buckingham Palace this morning to request the dissolution of parliament, setting the stage for a summer election. Labour currently holds a 22‑point lead in opinion polls, with the Conservatives struggling to recover from the Rwanda policy collapse and a stagnant economy. Starmer said, 'The country needs a mandate to deliver real change – clean energy, NHS reform, and re‑building ties with Europe.' Rishi Sunak accused Starmer of 'running away from scrutiny' and claimed Labour would raise taxes by £2,000 per household. The campaign is expected to focus on immigration, the cost of living, and Brexit."
        },
        {
            "url": "https://www.bbc.com/audio/play/w3ct8mbt",
            "text": "Audio: The Global Story – 'Europe's defence awakening'. In this 30‑minute podcast, the BBC's security correspondent Gordon Corera examines how Russia's war on Ukraine and Trump's threats to abandon NATO have forced Europe to rearm. Germany's Zeitenwende (turning point) has seen a €100 billion military fund; France has proposed a European nuclear deterrent; and Poland is spending 4% of GDP on defence – the highest in NATO. But can Europe truly defend itself without the US? Interviews with EU defence ministers and former NATO secretaries general suggest a 'two‑track' future: a weaker NATO alongside a more assertive European pillar. Listen on BBC Sounds."
        },
        {
            "url": "https://www.bbc.com/news/articles/c62jgzlwx09o",
            "text": "Brazil's Lula announces 'Green Marshall Plan' to save the Amazon. President Lula da Silva unveiled a $200 billion, 10‑year plan to reforest 150,000 square kilometres of the Amazon – an area the size of England – while creating 2 million jobs in sustainable industries. The plan is funded by a new global tax on oil companies and a debt‑for‑nature swap with Norway and Germany. Deforestation in the Brazilian Amazon had dropped 45% under Lula, but illegal gold mining and cattle ranching persist. 'We cannot do it alone,' Lula said at the UN climate summit. 'Rich countries that burned fossil fuels for a century must pay their share.'"
        }
    ],
    "sport": [
        {
            "url": "https://www.bbc.com/sport/football/articles/cz670q6583jo",
            "text": "Virgil van Dijk: Liverpool captain says Reds 'gave up' in FA Cup exit to Manchester City. Liverpool captain Virgil van Dijk has apologised to the club's supporters and said his side 'gave up' as they were dumped out of the FA Cup by Manchester City. The Reds' hopes of winning silverware this season now rest on Champions League glory following Saturday's humbling 4-0 quarter-final defeat at Etihad Stadium. 'I can only apologise to the fans for what we have shown, especially the second half,' said Netherlands centre-back Van Dijk, 34. 'We let our fans down, we let ourselves down, and the manager. The way we played in the second half, especially, must hurt for everyone. It definitely hurts me.'"
        },
        {
            "url": "https://www.bbc.com/sport/golf/articles/c98kn4996x6o",
            "text": "Masters 2026: Rory McIlroy completes career Grand Slam with thrilling one‑shot win at Augusta. McIlroy, 36, shot a final‑round 67 to hold off Scottie Scheffler and a charging Jon Rahm. The Northern Irishman, who had twice finished runner‑up at the Masters, famously collapsed at Augusta in 2011. 'I thought about that day every April for 15 years,' said an emotional McIlroy after holing a 15‑foot birdie on the 72nd green. He joins Tiger Woods, Jack Nicklaus, and Gene Sarazen as the only players to win all four majors. The victory also reunites him with the PGA Championship and The Open titles he won a decade ago."
        },
        {
            "url": "https://www.bbc.com/sport/football/articles/cp86v8zq4jeo",
            "text": "Manchester United confirm Erik ten Hag departure after Champions League failure. The Dutch coach leaves Old Trafford by mutual consent following a 5th‑place Premier League finish and a group‑stage exit from Europe. 'We thank Erik for his two trophies (Carabao Cup 2024, FA Cup 2025), but results this season have been inconsistent,' said a club statement. Former Chelsea manager Mauricio Pochettino is the early favourite to replace him, with Brentford's Thomas Frank also linked. United finished 22 points behind champions Arsenal. Captain Bruno Fernandes told club media: 'We players take responsibility – we underperformed.'"
        },
        {
            "url": "https://www.bbc.com/sport/formula1/articles/c1j7zn503eko",
            "text": "F1: Max Verstappen wins Australian GP after late safety car drama. The Red Bull driver took his third consecutive victory to open the 2026 season, but only after a crash‑induced safety car with five laps to go allowed Lewis Hamilton (Mercedes) to close a 12‑second gap. Verstappen held on to win by 0.8 seconds, with Ferrari's Charles Leclerc third. 'My tyres were gone, but we did just enough,' said Verstappen. Hamilton, 41, who announced this will be his final season, said: 'I gave it everything – maybe too much – but Max is a deserving winner.' Next race: Saudi Arabia in two weeks."
        },
        {
            "url": "https://www.bbc.com/sport/football/articles/cp9jvpl3m8go",
            "text": "Women's Euro 2026: England drawn with France, Sweden, and Portugal in group of death. Host nation England will face three top‑10 ranked teams in the group stage, starting with France at Wembley on 6 July. 'It's the toughest group possible, but if you want to win the tournament, you have to beat the best,' said Lionesses manager Sarina Wiegman. The other groups: Spain (holders) meet Germany, Italy, and Norway; Netherlands face USA, Denmark, and Switzerland. The tournament runs from 3 to 27 July across eight English cities. Tickets go on sale next Monday."
        },
        {
            "url": "https://www.bbc.com/sport/athletics/articles/c5yvpmvdd1ro",
            "text": "Kelvin Kiptum breaks own marathon world record in London – 2:00:35. The Kenyan runner, 26, shaved 25 seconds off his previous best from Chicago 2025, finishing just 35 seconds outside the mythical two‑hour barrier. Kiptum ran a negative split (60:47 first half, 59:48 second half), pulling away from Ethiopia's Sisay Lemma after 35 kilometres. 'I felt so strong; the crowd carried me,' Kiptum said. Britain's Emile Cairess finished seventh in a personal best 2:05:12. The women's race was won by Tigst Assefa of Ethiopia in 2:15:48, also a course record."
        }
    ],
    "tech": [
        {
            "url": "https://www.bbc.com/news/articles/ckgxdn9lyd2o",
            "text": "Apple at 50: Where did the tech giant succeed and fail? Apple celebrated its 50th birthday this week. The company has had some truly standout successes - and some notable flops. These days, nearly one out of every three people on the planet owns an Apple product. Among the hits: the iPod, which changed digital music forever; the iPhone, with more than 200 million sold each year; and the Apple Watch, which generates roughly $15bn annually. Among the misses: the Apple Lisa, a PC that was groundbreaking but 'far too costly'; the 'butterfly' keyboard design, a 'rare misstep in reliability'; and the Vision Pro headset, which was ultimately too 'cumbersome' and lacking in content to match the success of Apple's other products."
        },
        {
            "url": "https://www.bbc.com/news/articles/c145enxln0go",
            "text": "Microsoft unveils 'Copilot+ PCs' with built‑in AI that runs locally. The new Surface devices and third‑party laptops from Dell, HP, and Lenovo feature a neural processing unit (NPU) capable of 40 trillion operations per second. This allows real‑time translation, advanced photo editing, and a 'Recall' feature that records everything you've ever seen on the PC – searchable by natural language. Privacy advocates have raised concerns about the Recall feature storing screenshots. Microsoft promises all data stays on device and is encrypted. Prices start at $999, shipping in June."
        },
        {
            "url": "https://www.bbc.com/audio/play/p0nbxvv0",
            "text": "Audio: Tech Life – 'The quantum computing race heats up'. In this 25‑minute podcast, the BBC's technology editor Zoe Kleinman visits IBM's quantum data centre in New York, where a 1,200‑qubit processor will be unveiled next month. Google and Chinese researchers also claim recent breakthroughs in error correction. 'We are five years away from useful quantum advantage for chemistry and materials science,' says IBM's director of quantum. But the same technology could break current encryption, leading to a 'Q‑day' that security agencies are preparing for. The podcast also covers the first quantum computer sold to a pharmaceutical company. Listen on BBC Sounds."
        },
        {
            "url": "https://www.bbc.com/news/articles/cy41n17e23go",
            "text": "Meta launches 'Threads 2.0' with federated sharing and end‑to‑encrypted DMs. The new version of the Twitter rival, announced at Meta's annual developer conference, allows users to follow people on Mastodon, WordPress, and other platforms using the ActivityPub protocol. 'We are opening the social graph,' said Mark Zuckerberg. The update also includes private messaging that is encrypted by default, and a 'creator fund' that pays users based on engagement, not ads. Early tests show 10 million people have switched on the federation option. However, critics worry about Meta's control despite the open protocol – 'They can still see everything you do,' said one privacy advocate."
        },
        {
            "url": "https://www.bbc.com/future/article/20240912-what-riddles-teach-us-about-the-human-mind",
            "text": "What riddles teach us about the human mind. Riddles have been used for millennia to sharpen wits, but modern neuroscience shows they also reveal how our brains solve problems through 'insight' rather than analysis. A study at Northwestern University asked participants to solve classic puzzles like 'What has keys but no locks? (a piano)'. fMRI scans showed a burst of gamma waves in the right anterior temporal lobe just before the solution popped into consciousness – the 'Aha! moment'. This insight pathway is distinct from step‑by‑step reasoning and can be trained. The article also explores how AI, like GPT‑4, struggles with riddles because they require common sense and metaphor. 'Riddles may be the last frontier for human uniqueness,' concludes the author."
        }
    ]
}
# Собираем все новости в один DataFrame для инференса
inference_data = []
for category, articles in real_news.items():
    for article in articles:
        inference_data.append({
            "true_class": category,
            "url": article["url"],
            "text": article["text"]
        })

df_inference = pd.DataFrame(inference_data)

# 9. Инференс на реальных новостях
# ML модель
ml_labels, ml_confidences = [], []
for text in df_inference["text"]:
    proba = ml_pipeline.predict_proba([text])[0]
    pred_label = ml_pipeline.classes_[proba.argmax()]
    ml_labels.append(pred_label)
    ml_confidences.append(float(proba.max()))

# BERT модель
bert_labels, bert_confidences = bert_pipeline.predict(df_inference["text"].tolist())
bert_confidences = bert_confidences.tolist()

# Добавляем результаты в DataFrame
df_inference["ml_label"] = ml_labels
df_inference["ml_conf"] = ml_confidences
df_inference["bert_label"] = bert_labels
df_inference["bert_conf"] = bert_confidences


# 10. Сравнение качества на реальных новостях

df_inference["ml_correct"] = df_inference["true_class"] == df_inference["ml_label"]
df_inference["bert_correct"] = df_inference["true_class"] == df_inference["bert_label"]

ml_accuracy_real = df_inference["ml_correct"].mean()
bert_accuracy_real = df_inference["bert_correct"].mean()

print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ ИНФЕРЕНСА НА 25 СВЕЖИХ НОВОСТЯХ BBC")
print("="*60)
print(df_inference[["true_class", "url", "ml_label", "ml_conf", "bert_label", "bert_conf"]].to_string(max_colwidth=80))

print("\n" + "="*60)
print("ТОЧНОСТЬ (ACCURACY) НА РЕАЛЬНЫХ НОВОСТЯХ:")
print(f"ML модель (TF-IDF + LogisticRegression): {ml_accuracy_real:.2%}")
print(f"DistilBERT модель:                        {bert_accuracy_real:.2%}")
print("="*60)

if bert_accuracy_real > ml_accuracy_real:
    print("\n✅ DistilBERT работает лучше на свежих новостях.")
elif ml_accuracy_real > bert_accuracy_real:
    print("\n✅ Классическая ML модель работает лучше на свежих новостях.")
else:
    print("\n➖ Модели показали одинаковую точность.")

# Дополнительно: по каждой категории
print("\n" + "="*60)
print("ДЕТАЛИЗАЦИЯ ПО КАТЕГОРИЯМ (количество правильных / всего):")
for cat in real_news.keys():
    sub = df_inference[df_inference["true_class"] == cat]
    ml_cat_acc = sub["ml_correct"].sum() / len(sub)
    bert_cat_acc = sub["bert_correct"].sum() / len(sub)
    print(f"{cat.capitalize():15} | ML: {ml_cat_acc:.0%} ({sub['ml_correct'].sum()}/{len(sub)}) | BERT: {bert_cat_acc:.0%} ({sub['bert_correct'].sum()}/{len(sub)})")

Mounted at /content/drive

Распределение меток в обучающем датасете:
label
sport            511
business         506
politics         417
tech             401
entertainment    386
Name: count, dtype: int64

=== ML MODEL (на тестовой выборке) ===
Accuracy: 0.9842696629213483
               precision    recall  f1-score   support

     business       0.98      0.98      0.98       101
entertainment       0.99      0.97      0.98        77
     politics       0.98      0.98      0.98        84
        sport       1.00      1.00      1.00       103
         tech       0.98      0.99      0.98        80

     accuracy                           0.98       445
    macro avg       0.98      0.98      0.98       445
 weighted avg       0.98      0.98      0.98       445



tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2221 [00:00<?, ? examples/s]

...

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



=== ОБУЧЕНИЕ DISTILBERT ===


Epoch,Training Loss,Validation Loss
1,No log,0.088160
2,No log,0.133872
3,No log,0.093634



=== ОЦЕНКА DISTILBERT на тестовой выборке ===


{'eval_loss': 0.09363369643688202, 'eval_runtime': 7.5582, 'eval_samples_per_second': 58.877, 'eval_steps_per_second': 3.705, 'epoch': 3.0}

РЕЗУЛЬТАТЫ ИНФЕРЕНСА НА 25 СВЕЖИХ НОВОСТЯХ BBC
       true_class                                                                              url       ml_label   ml_conf     bert_label  bert_conf
0        business  https://www.bbc.com/worklife/article/20260331-dubais-tourism-industry-reels-...       business  0.391507       business   0.996520
1        business                                   https://www.bbc.com/news/articles/cn43wllgn4vo       business  0.613742       business   0.997174
2        business                                   https://www.bbc.com/news/articles/cvg49gq9y16o       business  0.701916       business   0.997272
3        business                                   https://www.bbc.com/news/articles/c70n2rjgxeyo       business  0.781867       business   0.997148
4        business                                   https://ww